## load and prepare data

In [ ]:
%cd ../..
%matplotlib inline

import numpy as np
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

import sys
sys.path.insert(0, 'evaluation_meldgraph')
from vol_eval_plots import (GROUP_3T, GROUP_7T_ADAPTED, GROUP_7T_DEFAULT, load_and_prepare_data,
                            harmo_labels, plot_on_surface, harmo_conditions_clusters,
                            surf_feature_names, NVERT, load_surf_feature_maps)


In [ ]:
eval_stats_df = load_and_prepare_data()

## read the surface features per vertex

In [ ]:
surf_feature_maps, cortex_mask, feature_rows = load_surf_feature_maps(eval_stats_df)

## compare the per-vertex differences between conditions

In [ ]:
# the conditions extracted above, compared vertex by vertex in the healthy controls: every
# comparison is one 7T reconstruction against 3T within one harmonisation, in the order that fixes
# the sign of the difference (the 7T group minus 3T), and the four of them are the columns of both
# figures below, unharmonised pair first
surf_difference_pairs = [(GROUP_3T, GROUP_7T_DEFAULT),
                         (GROUP_3T, GROUP_7T_ADAPTED)]

surf_difference_pair_labels = [((group_a, group_b, harmo),
                                f'{group_b} {harmo_labels[harmo]}\n- {group_a} {harmo_labels[harmo]}')
                               for harmo in harmo_conditions_clusters
                               for group_a, group_b in surf_difference_pairs]

# only healthy controls
control_subjects = sorted(eval_stats_df.loc[eval_stats_df['group'] == 'control',
                                            'site_subj_id'].unique())

median_difference_maps = {}
mad_difference_maps = {}
difference_size_records = []
for (group_a, group_b, harmo), label in surf_difference_pair_labels:
    for feature in surf_feature_names:
        # one comparison and feature at a time
        # only over the cortex label
        # hemispheres of a subject are averaged into one difference
        #    (subjects are statistically independent)
        subject_differences = {}
        for subject in control_subjects:
            maps_a = surf_feature_maps.get((harmo, group_a, subject, feature))
            maps_b = surf_feature_maps.get((harmo, group_b, subject, feature))
            if maps_a is None or maps_b is None:
                continue

            # mean over the two hemispheres
            maps_a = np.nanmean(maps_a, axis=0)
            maps_b = np.nanmean(maps_b, axis=0)

            difference = (maps_b - maps_a)[cortex_mask]
            if not np.isfinite(difference).any():
                continue

            # mean over the two hemispheres
            subject_differences[subject] = difference

        print(f'{label.replace(chr(10), " ")}, {surf_feature_names[feature]}: '
              f'{len(subject_differences)} control subjects')
        if not subject_differences:
            continue

        differences = np.stack(list(subject_differences.values()))

        # compute Wilcoxon signed-rank test per vertex, to see whether the difference is significantly different from zero
        # correct for multiple comparisons using FDR
        p_values = wilcoxon(differences, axis=0).pvalue
        p_values_corrected = multipletests(p_values, method='fdr_bh', alpha=0.05)[1]

        p_values_corrected_full = np.full(NVERT, np.nan)
        p_values_corrected_full[cortex_mask] = p_values_corrected

        # mean over all subjects per vertex
        diff_map = np.full(NVERT, np.nan)
        diff_map[cortex_mask] = np.nanmedian(differences, axis=0)
        diff_map[p_values_corrected_full > 0.05] = np.nan  # mask non-significant vertices
        median_difference_maps[((group_a, group_b, harmo), feature)] = diff_map

        # std over all subjects
        mad_map = np.full(NVERT, np.nan)
        from scipy.stats import median_abs_deviation
        mad_map[cortex_mask] = median_abs_deviation(differences, axis=0)   #np.nanstd(differences, axis=0)
        mad_difference_maps[((group_a, group_b, harmo), feature)] = mad_map

        # the same difference reduced the other way round, over the cortex of one subject instead
        # of over the subjects at one vertex. the absolute difference is averaged and not the
        # signed one
        difference_size_records += [
            {'comparison': f'{group_b} - {group_a}',
             'analysis_condition': harmo_labels[harmo],
             'feature': surf_feature_names[feature],
             'site_subj_id': subject,
             'mean_abs_difference': np.nanmean(np.abs(subject_difference))}
            for subject, subject_difference in subject_differences.items()]

# comparisons no subject contributed to are dropped, so that the figures below only have columns
# there is something to draw in
surf_difference_pair_labels = [(key, label) for key, label in surf_difference_pair_labels
                               if (key, next(iter(surf_feature_names))) in median_difference_maps]



### Figure 5, Supplementary Figure 6

In [ ]:
# two maps per feature, with the four comparisons as their columns: the
# mean difference over control subjects on a symmetric diverging scale, and its standard deviation
# over the same subjects on a sequential one starting at zero. colorscale shared
# across all features.
for difference_maps, cmap, symmetric, map_label in [
        (median_difference_maps, 'RdBu_r', True, 'Mean difference'),
        (mad_difference_maps, 'magma', False, 'Median absolute deviation of the difference')]:
    vmax = np.nanpercentile([np.abs(difference_maps[(key, feature)])
                             for key, _ in surf_difference_pair_labels
                             for feature in surf_feature_names], 99)
    # round vmax to two significant digits + 0.1, so that the colorbar ticks are nice numbers
    vmax = np.round(vmax, -int(np.floor(np.log10(vmax))) + 1) + 0.1
    print(f'Normalised features, {map_label.lower()}, colour scale '
          f'{-vmax if symmetric else 0:.3g} to {vmax:.3g}, columns left to right:')
    for column, (_, label) in enumerate(surf_difference_pair_labels, start=1):
        print(f'  {column}. ' + label.replace('\n', ' '))

    for feature, feature_label in surf_feature_names.items():
        print(feature_label)
        p = plot_on_surface([difference_maps[(key, feature)]
                             for key, _ in surf_difference_pair_labels],
                            [label for _, label in surf_difference_pair_labels],
                            cmap=cmap,
                            vmin=-vmax if symmetric else 0,
                            vmax=vmax,
                            continuous=True,
                            nan_color='lightgrey',
                            cbar_label=f'{map_label} in {feature_label} [z-score units]')
        p.show()
